# 核心独占调度系统性能深度分析报告 (Science Style)

本报告针对**核心独占 (Core Pinning)** 模式下的高频任务调度器进行定量分析。我们通过追踪任务从“唤醒”到“休眠”的完整物理生命周期，评估系统在**确定性 (Determinism)**、**负载均衡**及**并行扩展性**方面的表现。

### 1. 全路径生命周期定义 (Full Lifecycle)
为了精确捕捉调度损耗，我们将每次任务执行拆解为以下五个阶段：
1. **Wakeup Latency ($L_{wake}$)**: 线程收到信号到开始从全局队列获取任务的物理延迟。
2. **Pack Overhead ($L_{pack}$)**: 任务分发、上下文切换及输入数据打包的开销。
3. **Effective Computation ($C_i$)**: 算法核心逻辑执行时间。**这是系统唯一的有效产出。**
4. **Post Overhead ($L_{post}$)**: 计算完成到结果路由、状态更新的开销。
5. **Sleep Latency ($L_{sleep}$)**: 任务结束后，线程返回调度循环并重新进入休眠的残余延迟。

### 2. 视觉规范说明
* **时间单位**: 宏观视图统一使用 **毫秒 (ms)**，微观分析使用 **微秒 (us)**。
* **配色方案**: 采用低饱和度学术色系。**深蓝灰 (#3B5998)** 代表有效计算，**暗红 (#A6192E)** 代表调度损耗。
* **数据过滤**: 针对抖动分析，提供 **P99 截断版**（展示稳定性）与 **全量版**（展示最坏情况执行时间 WCET）。

In [19]:
import pandas as pd
import numpy as np
import plotly.express as px
import re, os


def _parse_hierarchy_data(csv):
    if not os.path.exists(csv):
        print(f"❌ File not found: {csv}")
        return pd.DataFrame()

    df = pd.read_csv(csv).sort_values(["tid", "seq"])
    df = df[df["t_us"] >= 0]

    excl = {"timer", "main::expand", "main::rollover"}
    rows = []
    warning_count = 0
    warning_details = []

    # 确定全局起始时间，用于计算毫秒偏移
    t_min_global = df["t_us"].min()

    for tid, g in df.groupby("tid"):
        events = g.reset_index(drop=True)
        for i in range(len(events)):
            curr = events.iloc[i]
            if curr["kind"] == "execute" and curr["tag"] not in excl:
                try:
                    prev_s = events.iloc[:i]
                    rel = prev_s[prev_s["kind"] == "release"].iloc[-1]
                    wak = prev_s[prev_s["kind"] == "wake"].iloc[-1]
                    next_s = events.iloc[i+1:]
                    com = next_s[next_s["kind"] == "complete"].iloc[0]
                    fin = next_s[next_s["kind"] == "finished"].iloc[0]
                    slp = next_s[next_s["kind"] == "sleep"].iloc[0]
                    temp_1 = next_s[next_s["kind"] == "temp_1"].iloc[0]
                    temp_2 = next_s[next_s["kind"] == "temp_2"].iloc[0]

                    # 按照用户要求分类计算：
                    wakeup_lat = rel["t_us"] - wak["t_us"]
                    pack = curr["t_us"] - rel["t_us"]
                    algo_us = com["t_us"] - curr["t_us"]
                    post = fin["t_us"] - com["t_us"]
                    sleep_lat = slp["t_us"] - fin["t_us"]
                    sleep_lat_1 = temp_1["t_us"] - fin["t_us"]
                    sleep_lat_2 = temp_2["t_us"] - temp_1["t_us"]
                    sleep_lat_3 = slp["t_us"] - temp_2["t_us"]


                    # === 警告逻辑：检查五个值是否小于零 ===
                    negative_values = {}
                    if wakeup_lat < 0:
                        negative_values['wakeup_lat'] = wakeup_lat
                    if pack < 0:
                        negative_values['pack'] = pack
                    if algo_us < 0:
                        negative_values['algo_us'] = algo_us
                    if post < 0:
                        negative_values['post'] = post
                    if sleep_lat < 0:
                        negative_values['sleep_lat'] = sleep_lat

                    if negative_values:
                        warning_count += 1
                        warning_details.append({
                            'tid': tid,
                            'algo': curr["tag"],
                            'seq': curr["seq"],
                            'negative_values': negative_values,
                            'timestamps': {
                                'wak': wak["t_us"],
                                'rel': rel["t_us"],
                                'curr': curr["t_us"],
                                'com': com["t_us"],
                                'fin': fin["t_us"],
                                'slp': slp["t_us"]
                            }
                        })
                        # 打印警告信息
                        print(f"\n⚠️  WARNING #{warning_count}: Negative value detected!")
                        print(f"   Thread: {tid}, Algorithm: {curr['tag']}, Seq: {curr['seq']}")
                        for key, val in negative_values.items():
                            print(f"   {key}: {val:.2f} us (negative)")
                        print(f"   Timeline: wake={wak['t_us']:.2f} → rel={rel['t_us']:.2f} → "
                              f"curr={curr['t_us']:.2f} → com={com['t_us']:.2f} → "
                              f"fin={fin['t_us']:.2f} → slp={slp['t_us']:.2f}")
                    # === 警告逻辑结束 ===

                    rows.append({
                        "tid": str(tid),
                        "algo": curr["tag"],
                        "algo_us": algo_us,
                        # 甘特图核心字段：起始点和持续时间（ms）
                        "start_ms": (wak["t_us"] - t_min_global) / 1000.0,
                        "algo_ms": algo_us / 1000.0,
                        # 统计字段
                        "algo_us": algo_us,
                        "wakeup_lat": wakeup_lat,
                        "sleep_lat": sleep_lat,
                        "sleep_lat_1": sleep_lat_1,
                        "sleep_lat_2": sleep_lat_2,
                        "sleep_lat_3": sleep_lat_3,
                        "pack": pack,
                        "post": post,
                        "pack_post_us": pack + post, # 算法级开销
                        "total_extra_us": (rel["t_us"] - wak["t_us"]) + pack + post + (slp["t_us"] - fin["t_us"]), # 全路径开销
                        "total_cycle_us": slp["t_us"] - wak["t_us"]
                    })
                except Exception as e:
                    # 打印异常信息以便调试
                    print(f"⚠️  Exception at tid={tid}, i={i}: {e}")
                    continue

    # === 在函数结束前打印汇总信息 ===
    print(f"\n{'='*60}")
    print(f"📊 PARSING COMPLETE")
    print(f"{'='*60}")
    print(f"Total events parsed: {len(rows)}")
    print(f"Total threads: {df['tid'].nunique()}")

    if warning_count > 0:
        print(f"\n⚠️  WARNINGS: {warning_count} negative value(s) detected!")
        print(f"   First 3 warnings:")
        for i, detail in enumerate(warning_details[:3]):
            print(f"   {i+1}. tid={detail['tid']}, algo={detail['algo']}, "
                  f"negatives={list(detail['negative_values'].keys())}")
        if warning_count > 3:
            print(f"   ... and {warning_count - 3} more warnings")
    else:
        print(f"\n✅ All values are non-negative. No warnings detected.")

    print(f"{'='*60}\n")

    return pd.DataFrame(rows)


# 执行解析
CSV_PATH = "temp/tracing.csv"
print(f" Reading CSV: {CSV_PATH}")
raw_df = pd.read_csv(CSV_PATH)
print(f" Raw data loaded: {len(raw_df)} rows")

analysis_df = _parse_hierarchy_data(CSV_PATH)
print(f" Analysis data created: {len(analysis_df)} rows")

# 显示数据预览
if not analysis_df.empty:
    print("\n Data preview (first 3 rows):")
    print(analysis_df[['tid', 'algo', 'wakeup_lat', 'pack', 'algo_us', 'post', 'sleep_lat']].head(3))

 Reading CSV: temp/tracing.csv
 Raw data loaded: 79608 rows

📊 PARSING COMPLETE
Total events parsed: 9700
Total threads: 8

✅ All values are non-negative. No warnings detected.

 Analysis data created: 9700 rows

 Data preview (first 3 rows):
     tid algo  wakeup_lat  pack  algo_us  post  sleep_lat
0  18705   n6           2     6     1227     4          9
1  18705  n26           2     3       89     6          4
2  18705  n28           1     1      173     2          4


## 一、 宏观并行行为分析 (Timeline & Gantt)

### 1. 任务甘特图 (Hardware Core Gantt Chart)
展示有效计算块 ($C_i$) 的持续时长。
* **分析重点**: 观察各核心条块之间的**横向间隙**。这些空隙（气泡）代表了调度器无法填满的资源空洞，是加速比无法达到线性的根本原因。

In [4]:
def draw_px_gantt(df_jobs):
    if df_jobs.empty:
        print("Warning: Input DataFrame is empty.")
        return

    # 1. 精选 Science 级低饱和度多色盘 (针对不同 Algo)
    # 选色原则：冷暖交替，灰度梯度一致
    DISCRETE_SCI_COLORS = [
        "#3B5998", # 灰蓝
        "#5E8B61", # 鼠尾草绿
        "#A6192E", # 砖红
        "#D6A531", # 暗金
        "#709BFF", # 浅钢蓝
        "#925E9F", # 灰紫
        "#0099B4", # 墨青
        "#FDAF91", # 柔粉
        "#4D4D4D", # 碳灰
        "#ADB6B6"  # 铝灰
    ]

    # 2. 线程 TID 自然排序 (确保 T0, T1, T2... 逻辑顺序)
    def extract_num(s):
        n = re.findall(r'\d+', str(s))
        return int(n[0]) if n else 0

    tids_sorted = sorted(df_jobs["tid"].unique(), key=extract_num)
    # 算法也进行排序，确保图例整齐
    algos_sorted = sorted(df_jobs["algo"].unique(), key=extract_num)

    # 3. 绘图
    fig = px.bar(
        df_jobs,
        base="start_ms",      # 任务起点 (ms)
        x="algo_ms",          # 任务宽度 (ms)
        y="tid",
        color="algo",         # 区分不同算法
        orientation='h',
        category_orders={"tid": tids_sorted, "algo": algos_sorted},
        color_discrete_sequence=DISCRETE_SCI_COLORS,
        title="Hardware Core Execution Continuity (Science Style)",
        labels={"start_ms": "Timeline", "algo_ms": "Exec", "tid": "Thread ID"}
    )

    # 4. 论文级布局优化
    fig.update_layout(
        plot_bgcolor='rgba(248, 248, 248, 1)', # 极浅灰背景，使白色气泡更明显
        xaxis=dict(
            title="Time Offset (ms)",
            showgrid=True,
            gridcolor='white', # 白色网格在灰底上非常高级
            linecolor='black',
            ticks='outside',
            zeroline=False
        ),
        yaxis=dict(
            title="Exclusive Thread / Core",
            showgrid=False, # Y 轴不需网格，保持横向视觉流
            autorange="reversed", # 线程 0 置顶
            linecolor='black',
            ticks='outside'
        ),
        font=dict(family="Arial", size=12, color="black"),
        height=350 + (len(tids_sorted) * 30), # 根据线程数动态调整高度
        margin=dict(l=100, r=30, t=100, b=80), # 增加左边距给 TID 标签
    )

    # 5. 交互与渲染优化
    fig.update_traces(
        marker_line_width=0.5, # 给极其微小的任务块增加细边框以便识别
        marker_line_color="rgba(255,255,255,0.2)",
        opacity=0.9,
        # 悬浮窗：微秒级精度标注
        hovertemplate="<b>Core %{y}</b><br>Algo: %{text}<br>Start: %{base:.3f} ms<br>Duration: %{x:.3f} ms<extra></extra>",
        text=df_jobs["algo"]
    )

    fig.show()

# 执行
draw_px_gantt(analysis_df)

In [5]:
# 1. Nature Classic - 包含所有类别
NATURE_CLASSIC = {
    # 四个阶段
    'wakeup_lat': '#3B7A9E',    # 沉稳蓝色
    'pack': '#6BA3B8',           # 柔和蓝绿
    "algo_us": "#E8E8E8",    # 【淡灰】 有效功设为背景
    'post': '#E8A87C',           # 暖橙
    'sleep_lat': '#C87D7D',      # 柔和红褐

    # 新增三类
    'Active': '#2C5F7A',         # 深海蓝 - 活跃状态
    'Overhead': '#D4834A',       # 橙褐 - 开销
    'Idle': '#8CAA8C',           # 柔和灰绿 - 空闲状态
}

# 2. Nature Earth - 大地色系
NATURE_EARTH = {
    'wakeup_lat': '#4A6B7F',    # 深蓝灰
    'pack': '#7A9B8C',           # 苔藓绿
    "algo_us": "#E8E8E8",    # 【淡灰】 有效功设为背景
    'post': '#C49A6C',           # 沙褐色
    'sleep_lat': '#B57D7D',      # 玫瑰棕

    'Active': '#3D5A6B',         # 深灰蓝
    'Overhead': '#B8896C',       # 暖棕
    'Idle': '#9AAB9A',           # 灰
}

# 3. Nature Ocean - 海洋色系
NATURE_OCEAN = {
    'wakeup_lat': '#2C5F7A',    # 深海蓝
    'pack': '#4A8C8C',           # 碧绿
    "algo_us": "#E8E8E8",    # 【淡灰】 有效功设为背景
    'post': '#D4A76A',           # 金色
    'sleep_lat': '#B06A6A',      # 珊瑚红

    'Active': '#1A4A6A',         # 深蓝
    'Overhead': '#C4956A',       # 暖金褐
    'Idle': '#6A9A9A',           # 海绿
}

# 4. Nature Journal - Nature期刊常见配色
NATURE_JOURNAL = {
    'wakeup_lat': '#1F567D',    # 深蓝
    'pack': '#5B8C5A',           # 森林绿
    "algo_us": "#E8E8E8",    # 【淡灰】 有效功设为背景
    'post': '#D4834A',           # 橙褐
    'sleep_lat': '#B55A5A',      # 砖红

    'Active': '#1A4A6E',         # 墨水蓝
    'Overhead': '#C47A4A',       # 陶土橙
    'Idle': '#7A9A7A',           # 灰绿
}

# 5. Nature Minimal - 极简灰度
NATURE_MINIMAL = {
    'wakeup_lat': '#4A4A4A',    # 深灰
    'pack': '#7A7A7A',           # 中灰
    "algo_us": "#E8E8E8",    # 【淡灰】 有效功设为背景
    'post': '#AAAAAA',           # 浅灰
    'sleep_lat': '#D4D4D4',      # 极浅灰

    'Active': '#333333',         # 最深灰
    'Overhead': '#666666',       # 中深灰
    'Idle': '#999999',           # 中灰
}

# 6. Nature Garden - 花园色系，温暖自然
NATURE_GARDEN = {
    'wakeup_lat': '#4A7A9C',    # 天蓝
    'pack': '#6A9B7A',           # 草绿
    "algo_us": "#E8E8E8",    # 【淡灰】 有效功设为背景
    'post': '#D4A06A',           # 麦黄
    'sleep_lat': '#C07A7A',      # 粉红

    'Active': '#3A6A8A',         # 深天蓝
    'Overhead': '#C4906A',       # 卡其
    'Idle': '#8AAA8A',           # 嫩绿
}

# 7. Nature Evening - 晚间色系，沉稳
NATURE_EVENING = {
    'wakeup_lat': '#3D5A7A',    # 暮蓝
    'pack': '#5A7A6A',           # 暮绿
    "algo_us": "#E8E8E8",    # 【淡灰】 有效功设为背景
    'post': '#B08A6A',           # 暮橙
    'sleep_lat': '#A06A6A',      # 暮红

    'Active': '#2A4A6A',         # 夜蓝
    'Overhead': '#A07A5A',       # 暮褐
    'Idle': '#6A8A7A',           # 暮灰绿
}

PLA = NATURE_CLASSIC


## 二、 核心资源利用效率分析 (Efficiency Breakdown)

本图表将单核独占的物理时间归一化为 **100%**，并切分为三个互斥状态：
1. **Active (有效功)**: 真正的算法产出时间占比。
2. **Sched Overhead (调度开销)**: 全路径（Wake+Pack+Post+Sleep）的非生产性耗时总和。
3. **Wait/Idle (空闲气泡)**: 核心处于 `cv_wait` 状态，等待任务流或同步信号的时间。

In [6]:
import re
import plotly.express as px
import pandas as pd

def draw_thread_analysis_mixed(df_raw, df_jobs):
    """
    修正版：
    1. 利用率堆叠柱状图：横向 (Horizontal)
    2. 额外开销小提琴图：纵向 (Vertical)
    """
    if df_jobs.empty:
        print("数据为空")
        return

    # --- 1. 数据准备：利用率百分比 ---
    wall_clock = df_raw['t_us'].max() - df_raw['t_us'].min()
    thread_res = []

    for tid in df_jobs['tid'].unique():
        sub = df_jobs[df_jobs['tid'] == tid]
        active_time = sub['algo_us'].sum()
        overhead_time = sub['total_extra_us'].sum()
        # Idle = 总时间 - (算法时间 + 调度总开销)
        idle_time = max(0, wall_clock - (active_time + overhead_time))

        thread_res.append({"tid": tid, "State": "1. Active", "Value": (active_time / wall_clock) * 100})
        thread_res.append({"tid": tid, "State": "2. Overhead", "Value": (overhead_time / wall_clock) * 100})
        thread_res.append({"tid": tid, "State": "3. Idle", "Value": (idle_time / wall_clock) * 100})

    df_util = pd.DataFrame(thread_res)
    # 线程号自然排序 (T0, T1, T2...)
    tids_sorted = sorted(df_jobs["tid"].unique(), key=lambda x: int(re.findall(r'\d+', str(x))[0]))

    # --- 2. 绘制【横向】堆叠利用率图 ---
    fig_util = px.bar(
        df_util,
        x="Value",        # X轴是百分比数值
        y="tid",          # Y轴是线程ID
        color="State",
        orientation='h',  # 横向
        category_orders={"tid": tids_sorted},
        color_discrete_map={
            "1. Active": PLA["Active"],
            "2. Overhead": PLA["Overhead"],
            "3. Idle": PLA["Idle"]
        },
        title="Thread-level Core Utilization Breakdown",
        labels={"Value": "Percentage of Time (%)", "tid": "Thread ID"}
    )

    fig_util.update_layout(
        plot_bgcolor='white',
        barmode='stack',
        xaxis=dict(range=[0, 105], gridcolor='#F0F0F0', ticksuffix="%", title="Core Utilization (%)"),
        yaxis=dict(autorange="reversed", linecolor='black', title="Thread ID"), # T0 在顶端
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=300 + (len(tids_sorted) * 30)
    )
    fig_util.show()

# 调用
draw_thread_analysis_mixed(raw_df, analysis_df)

## 三、 各算法的唤醒、包装、分发效率分析
1. 手风琴式堆叠柱状图 (Accordion Stacked Bar Chart): 展示每个算法的唤醒、包装、分发、计算、后处理、休眠的时间占比。

In [7]:
import re
import plotly.express as px
import pandas as pd
import numpy as np

def draw_px_accordion_enhanced_us(df_jobs):
    """
    强化开销显示的风琴图 (单位回归为 us)
    """
    if df_jobs.empty:
        print("数据为空")
        return

    # 1. 均值计算 (直接使用原始 us 单位)
    cols = ["wakeup_lat", "pack", "algo_us", "post", "sleep_lat"]
    g = df_jobs.groupby("algo")[cols].mean().reset_index()

    # 2. 自然排序 (n0, n1, n2...)
    def extract_num(s):
        n = re.findall(r'\d+', str(s))
        return int(n[0]) if n else 0
    g['sort_idx'] = g['algo'].apply(extract_num)
    g = g.sort_values("sort_idx", ascending=False)

    # 3. 准备百分比数据 (归一化)
    g_pct = g.copy()
    g_pct['total'] = g_pct[cols].sum(axis=1)
    for col in cols:
        g_pct[col] = (g_pct[col] / g_pct['total']) * 100

    # --- 绘图 A：绝对耗时构成 (us) ---
    long_abs = g.melt(id_vars="algo", value_vars=cols, var_name="Stage", value_name="us")
    fig_abs = px.bar(
        long_abs, y="algo", x="us", color="Stage",
        orientation='h',
        color_discrete_map=PLA,
        title="Mean Task Lifecycle Breakdown (Absolute us)",
        labels={"us": "Time Duration (us)", "algo": "Algorithm Node"}
    )

    # 移除右侧的 Sched Overhead 标注
    # for i in range(len(g)):
    #     row = g.iloc[i]
    #     overhead = row["wakeup_lat"] + row["pack"] + row["post"] + row["sleep_lat"]
    #     fig_abs.add_annotation(
    #         x=row[cols].sum(),
    #         y=row["algo"],
    #         text=f" Sched:{overhead:.1f}us",
    #         showarrow=False,
    #         xanchor="left",
    #         font=dict(color="#A6192E", size=10, family="Arial Black")
    #     )

    fig_abs.update_layout(
        plot_bgcolor='white',
        barmode='stack',
        height=len(g)*35+150,
        xaxis=dict(gridcolor='#F0F0F0', title="Average Time (us)")
    )
    fig_abs.show()

    # --- 绘图 B：结构占比构成 (100% 归一化) ---
    long_pct = g_pct.melt(id_vars="algo", value_vars=cols, var_name="Stage", value_name="pct")
    fig_pct = px.bar(
        long_pct, y="algo", x="pct", color="Stage",
        orientation='h',
        color_discrete_map=PLA,
        title="Task Lifecycle Structural Ratio (Normalized %)",
        labels={"pct": "Composition Ratio (%)", "algo": "Algorithm Node"}
        # 删除 text_auto='.1f' 这行，不显示数据标签
    )

    fig_pct.update_layout(
        plot_bgcolor='white',
        barmode='stack',
        xaxis=dict(title="Percentage (%)", range=[0, 100], ticksuffix="%", gridcolor='#F0F0F0'),
        height=len(g)*35+150,
        legend_title="Lifecycle Stages"
    )
    fig_pct.show()

# 调用
draw_px_accordion_enhanced_us(analysis_df)

## 四、额外开销的真实分布
1. **小提琴图 (Violin Plot)**: 展示每个线程的调度开销分布，重点关注均值。
2. **小提琴图 (Violin Plot)**: 展示每个算法的调度开销分布，重点关注均值。
分别提供全量与p99截断两种视图，便于分析调度器的稳定性与最坏情况执行时间 (WCET)。

In [8]:
def draw_system_summary_violin(df_jobs):
    # 创建统一标签
    df_sum = df_jobs.copy()
    df_sum["System"] = "All Tasks"

    fig = px.violin(df_sum, x="System", y="total_extra_us",
                    box=True, points="outliers",
                    title="System-wide Total Extra Overhead Distribution",
                    color_discrete_sequence=[PLA["Overhead"]])

    # 增加均值和 P99 标注
    mean_val = df_sum["total_extra_us"].mean()
    p99_val = df_sum["total_extra_us"].quantile(0.99)

    fig.add_hline(y=mean_val, line_dash="dot", annotation_text=f"Mean: {mean_val:.1f}us")
    fig.add_hline(y=p99_val, line_dash="dash", line_color="red",
                  annotation_text=f"P99: {p99_val:.1f}us", annotation_position="top left")
    fig.update_yaxes(range=[0, None])  # 从0开始
    fig.update_layout(plot_bgcolor='white', yaxis_title="Total Extra Overhead (us)")
    fig.show()

draw_system_summary_violin(analysis_df)

In [27]:
# Cell 1: Wakeup Latency Distribution by Algorithm
def draw_algo_wakeup_violin(df, p99_cut=False):
    """
    Draw algorithm-level Wakeup Latency distribution
    Wakeup Latency: wake → release
    """
    metric = 'wakeup_lat'
    metric_name = 'Wakeup Latency (wake→release)'

    df_plot = df.copy()
    if p99_cut:
        p99_threshold = df_plot[metric].quantile(0.99)
        df_plot = df_plot[df_plot[metric] <= p99_threshold]
        suffix = " (P99 Truncated)"
    else:
        suffix = ""

    fig = px.violin(
        df_plot,
        x='algo',
        y=metric,
        box=True,
        points="outliers",
        title=f"Algorithm {metric_name} Distribution{suffix}",
        color_discrete_sequence=[PLA['wakeup_lat']],  # 直接取单个颜色，注意是列表
        labels={'algo': 'Algorithm', metric: f'{metric_name} (µs)'}
    )

    mean_val = df_plot[metric].mean()
    median_val = df_plot[metric].median()
    p99_val = df_plot[metric].quantile(0.99)

    fig.add_hline(y=mean_val, line_dash="dot", line_color="black",
                  annotation_text=f"Mean: {mean_val:.1f}µs", annotation_position="bottom right")
    fig.add_hline(y=p99_val, line_dash="dash", line_color="red",
                  annotation_text=f"P99: {p99_val:.1f}µs", annotation_position="top left")
    fig.add_hline(y=median_val, line_dash="dashdot", line_color="blue",
                  annotation_text=f"Median: {median_val:.1f}µs", annotation_position="bottom left")

    fig.update_layout(plot_bgcolor='white', height=500, width=900)
    fig.update_yaxes(gridcolor='lightgray', gridwidth=0.5, zeroline=True, zerolinecolor='lightgray', range=[0, None])
    fig.show()

    print(f"\nWakeup Latency - Mean: {mean_val:.1f}µs, P99: {p99_val:.1f}µs, Samples: {len(df_plot)}")

draw_algo_wakeup_violin(analysis_df, p99_cut=False)


Wakeup Latency - Mean: 1.7µs, P99: 10.0µs, Samples: 9700


In [29]:
# Cell 2: Pack Overhead Distribution by Algorithm
def draw_algo_pack_violin(df, p99_cut=False):
    """
    Draw algorithm-level Pack Overhead distribution
    Pack: release → execute
    """
    metric = 'pack'
    metric_name = 'Pack Overhead (release→execute)'

    df_plot = df.copy()
    if p99_cut:
        p99_threshold = df_plot[metric].quantile(0.99)
        df_plot = df_plot[df_plot[metric] <= p99_threshold]
        suffix = " (P99 Truncated)"
    else:
        suffix = ""

    fig = px.violin(
        df_plot,
        x='algo',
        y=metric,
        box=True,
        points="outliers",
        title=f"Algorithm {metric_name} Distribution{suffix}",
        color_discrete_sequence=[PLA['pack']],  # 直接取单个颜色，注意是列表
        labels={'algo': 'Algorithm', metric: f'{metric_name} (µs)'}
    )

    mean_val = df_plot[metric].mean()
    median_val = df_plot[metric].median()
    p99_val = df_plot[metric].quantile(0.99)

    fig.add_hline(y=mean_val, line_dash="dot", line_color="black",
                  annotation_text=f"Mean: {mean_val:.1f}µs", annotation_position="bottom right")
    fig.add_hline(y=p99_val, line_dash="dash", line_color="red",
                  annotation_text=f"P99: {p99_val:.1f}µs", annotation_position="top left")
    fig.add_hline(y=median_val, line_dash="dashdot", line_color="blue",
                  annotation_text=f"Median: {median_val:.1f}µs", annotation_position="bottom left")

    fig.update_layout(plot_bgcolor='white', height=500, width=900)
    fig.update_yaxes(gridcolor='lightgray', gridwidth=0.5, zeroline=True, zerolinecolor='lightgray', range=[0, None])
    fig.show()

    print(f"\nPack Overhead - Mean: {mean_val:.1f}µs, P99: {p99_val:.1f}µs, Samples: {len(df_plot)}")

draw_algo_pack_violin(analysis_df, p99_cut=False)


Pack Overhead - Mean: 2.6µs, P99: 12.0µs, Samples: 9700


In [30]:
# Cell 3: Post Overhead Distribution by Algorithm
def draw_algo_post_violin(df, p99_cut=False):
    """
    Draw algorithm-level Post Overhead distribution
    Post: complete → finished
    """
    metric = 'post'
    metric_name = 'Post Overhead (complete→finished)'

    df_plot = df.copy()
    if p99_cut:
        p99_threshold = df_plot[metric].quantile(0.99)
        df_plot = df_plot[df_plot[metric] <= p99_threshold]
        suffix = " (P99 Truncated)"
    else:
        suffix = ""

    fig = px.violin(
        df_plot,
        x='algo',
        y=metric,
        box=True,
        points="outliers",
        title=f"Algorithm {metric_name} Distribution{suffix}",
        color_discrete_sequence=[PLA['post']],  # 直接取单个颜色，注意是列表
        labels={'algo': 'Algorithm', metric: f'{metric_name} (µs)'}
    )

    mean_val = df_plot[metric].mean()
    median_val = df_plot[metric].median()
    p99_val = df_plot[metric].quantile(0.99)

    fig.add_hline(y=mean_val, line_dash="dot", line_color="black",
                  annotation_text=f"Mean: {mean_val:.1f}µs", annotation_position="bottom right")
    fig.add_hline(y=p99_val, line_dash="dash", line_color="red",
                  annotation_text=f"P99: {p99_val:.1f}µs", annotation_position="top left")
    fig.add_hline(y=median_val, line_dash="dashdot", line_color="blue",
                  annotation_text=f"Median: {median_val:.1f}µs", annotation_position="bottom left")

    fig.update_layout(plot_bgcolor='white', height=500, width=900)
    fig.update_yaxes(gridcolor='lightgray', gridwidth=0.5, zeroline=True, zerolinecolor='lightgray', range=[0, None])
    fig.show()

    print(f"\nPost Overhead - Mean: {mean_val:.1f}µs, P99: {p99_val:.1f}µs, Samples: {len(df_plot)}")

draw_algo_post_violin(analysis_df, p99_cut=False)


Post Overhead - Mean: 4.8µs, P99: 13.0µs, Samples: 9700


In [31]:
# Cell 4: Sleep Latency Distribution by Algorithm
def draw_algo_sleep_violin(df, p99_cut=False):
    """
    Draw algorithm-level Sleep Latency distribution
    Sleep Latency: finished → sleep
    """
    metric = 'sleep_lat'
    metric_name = 'Sleep Latency (finished→sleep)'

    df_plot = df.copy()
    if p99_cut:
        p99_threshold = df_plot[metric].quantile(0.99)
        df_plot = df_plot[df_plot[metric] <= p99_threshold]
        suffix = " (P99 Truncated)"
    else:
        suffix = ""

    fig = px.violin(
        df_plot,
        x='algo',
        y=metric,
        box=True,
        points="outliers",
        title=f"Algorithm {metric_name} Distribution{suffix}",
        color_discrete_sequence=[PLA['sleep_lat']],  # 直接取单个颜色，注意是列表
        labels={'algo': 'Algorithm', metric: f'{metric_name} (µs)'}
    )

    mean_val = df_plot[metric].mean()
    median_val = df_plot[metric].median()
    p99_val = df_plot[metric].quantile(0.99)

    fig.add_hline(y=mean_val, line_dash="dot", line_color="black",
                  annotation_text=f"Mean: {mean_val:.1f}µs", annotation_position="bottom right")
    fig.add_hline(y=p99_val, line_dash="dash", line_color="red",
                  annotation_text=f"P99: {p99_val:.1f}µs", annotation_position="top left")
    fig.add_hline(y=median_val, line_dash="dashdot", line_color="blue",
                  annotation_text=f"Median: {median_val:.1f}µs", annotation_position="bottom left")

    fig.update_layout(plot_bgcolor='white', height=500, width=900)
    fig.update_yaxes(gridcolor='lightgray', gridwidth=0.5, zeroline=True, zerolinecolor='lightgray', range=[0, None])
    fig.show()

    print(f"\nSleep Latency - Mean: {mean_val:.1f}µs, P99: {p99_val:.1f}µs, Samples: {len(df_plot)}")

draw_algo_sleep_violin(analysis_df, p99_cut=False)


Sleep Latency - Mean: 6.7µs, P99: 28.0µs, Samples: 9700


In [32]:
# Cell 4: Sleep Latency 1 Distribution by Algorithm
def draw_algo_sleep_violin(df, p99_cut=False):
    """
    Draw algorithm-level Sleep Latency distribution
    Sleep Latency: finished → sleep
    """
    metric = 'sleep_lat_1'
    metric_name = 'Sleep Latency (finished→sleep)'

    df_plot = df.copy()
    if p99_cut:
        p99_threshold = df_plot[metric].quantile(0.99)
        df_plot = df_plot[df_plot[metric] <= p99_threshold]
        suffix = " (P99 Truncated)"
    else:
        suffix = ""

    fig = px.violin(
        df_plot,
        x='algo',
        y=metric,
        box=True,
        points="outliers",
        title=f"Algorithm {metric_name} Distribution{suffix}",
        color_discrete_sequence=[PLA['sleep_lat']],  # 直接取单个颜色，注意是列表
        labels={'algo': 'Algorithm', metric: f'{metric_name} (µs)'}
    )

    mean_val = df_plot[metric].mean()
    median_val = df_plot[metric].median()
    p99_val = df_plot[metric].quantile(0.99)

    fig.add_hline(y=mean_val, line_dash="dot", line_color="black",
                  annotation_text=f"Mean: {mean_val:.1f}µs", annotation_position="bottom right")
    fig.add_hline(y=p99_val, line_dash="dash", line_color="red",
                  annotation_text=f"P99: {p99_val:.1f}µs", annotation_position="top left")
    fig.add_hline(y=median_val, line_dash="dashdot", line_color="blue",
                  annotation_text=f"Median: {median_val:.1f}µs", annotation_position="bottom left")

    fig.update_layout(plot_bgcolor='white', height=500, width=900)
    fig.update_yaxes(gridcolor='lightgray', gridwidth=0.5, zeroline=True, zerolinecolor='lightgray', range=[0, None])
    fig.show()

    print(f"\nSleep Latency 1 - Mean: {mean_val:.1f}µs, P99: {p99_val:.1f}µs, Samples: {len(df_plot)}")

draw_algo_sleep_violin(analysis_df, p99_cut=False)


Sleep Latency 1 - Mean: 1.5µs, P99: 14.0µs, Samples: 9700


In [33]:
# Cell 4: Sleep Latency 2 Distribution by Algorithm
def draw_algo_sleep_violin(df, p99_cut=False):
    """
    Draw algorithm-level Sleep Latency distribution
    Sleep Latency: finished → sleep
    """
    metric = 'sleep_lat_2'
    metric_name = 'Sleep Latency (finished→sleep)'

    df_plot = df.copy()
    if p99_cut:
        p99_threshold = df_plot[metric].quantile(0.99)
        df_plot = df_plot[df_plot[metric] <= p99_threshold]
        suffix = " (P99 Truncated)"
    else:
        suffix = ""

    fig = px.violin(
        df_plot,
        x='algo',
        y=metric,
        box=True,
        points="outliers",
        title=f"Algorithm {metric_name} Distribution{suffix}",
        color_discrete_sequence=[PLA['sleep_lat']],  # 直接取单个颜色，注意是列表
        labels={'algo': 'Algorithm', metric: f'{metric_name} (µs)'}
    )

    mean_val = df_plot[metric].mean()
    median_val = df_plot[metric].median()
    p99_val = df_plot[metric].quantile(0.99)

    fig.add_hline(y=mean_val, line_dash="dot", line_color="black",
                  annotation_text=f"Mean: {mean_val:.1f}µs", annotation_position="bottom right")
    fig.add_hline(y=p99_val, line_dash="dash", line_color="red",
                  annotation_text=f"P99: {p99_val:.1f}µs", annotation_position="top left")
    fig.add_hline(y=median_val, line_dash="dashdot", line_color="blue",
                  annotation_text=f"Median: {median_val:.1f}µs", annotation_position="bottom left")

    fig.update_layout(plot_bgcolor='white', height=500, width=900)

    fig.update_yaxes(gridcolor='lightgray', gridwidth=0.5, zeroline=True, zerolinecolor='lightgray', range=[0, None])
    fig.show()

    print(f"\nSleep Latency 2 - Mean: {mean_val:.1f}µs, P99: {p99_val:.1f}µs, Samples: {len(df_plot)}")

draw_algo_sleep_violin(analysis_df, p99_cut=False)


Sleep Latency 2 - Mean: 2.6µs, P99: 9.0µs, Samples: 9700


In [34]:
# Cell 4: Sleep Latency 2 Distribution by Algorithm
def draw_algo_sleep_violin(df, p99_cut=False):
    """
    Draw algorithm-level Sleep Latency distribution
    Sleep Latency: finished → sleep
    """
    metric = 'sleep_lat_3'
    metric_name = 'Sleep Latency (finished→sleep)'

    df_plot = df.copy()
    if p99_cut:
        p99_threshold = df_plot[metric].quantile(0.99)
        df_plot = df_plot[df_plot[metric] <= p99_threshold]
        suffix = " (P99 Truncated)"
    else:
        suffix = ""

    fig = px.violin(
        df_plot,
        x='algo',
        y=metric,
        box=True,
        points="outliers",
        title=f"Algorithm {metric_name} Distribution{suffix}",
        color_discrete_sequence=[PLA['sleep_lat']],  # 直接取单个颜色，注意是列表
        labels={'algo': 'Algorithm', metric: f'{metric_name} (µs)'}
    )

    mean_val = df_plot[metric].mean()
    median_val = df_plot[metric].median()
    p99_val = df_plot[metric].quantile(0.99)

    fig.add_hline(y=mean_val, line_dash="dot", line_color="black",
                  annotation_text=f"Mean: {mean_val:.1f}µs", annotation_position="bottom right")
    fig.add_hline(y=p99_val, line_dash="dash", line_color="red",
                  annotation_text=f"P99: {p99_val:.1f}µs", annotation_position="top left")
    fig.add_hline(y=median_val, line_dash="dashdot", line_color="blue",
                  annotation_text=f"Median: {median_val:.1f}µs", annotation_position="bottom left")

    fig.update_layout(plot_bgcolor='white', height=500, width=900)
    fig.update_yaxes(gridcolor='lightgray', gridwidth=0.5, zeroline=True, zerolinecolor='lightgray', range=[0, None])
    fig.show()

    print(f"\nSleep Latency 3 - Mean: {mean_val:.1f}µs, P99: {p99_val:.1f}µs, Samples: {len(df_plot)}")

draw_algo_sleep_violin(analysis_df, p99_cut=False)


Sleep Latency 3 - Mean: 2.6µs, P99: 17.0µs, Samples: 9700


## 五、 任务负载结构分析 (Structural Composition)

本部分通过双维度指标评估任务集的特征：
* **Task Frequency Ratio ($P_i$)**: 任务发生的频次分布。高频小任务往往是调度开销（Overhead）累积的主要来源。
* **Effective Workload Ratio ($P_i \times C_i$)**: 任务对 CPU 总时间的实际占用贡献。用于识别系统中的“重型节点”，即 Amahdl 定律中的串行瓶颈或关键路径。

# 加速比与扩展性分析 (Speedup & Scalability)

### 1. 核心定义
* **Speedup $S(m)$**: $S(m) = \frac{T_{hp}(1)}{T_{hp}(m)}$
  - $T_{hp}(1)$: 单核心运行整个超周期（Makespan）的中位时间。
  - $T_{hp}(m)$: $m$ 个核心并行运行时，完成同样任务量的时间。
* **Parallel Efficiency $E(m)$**: $E(m) = \frac{S(m)}{m} \times 100\%$
  - 反映了每增加一个核心，带来的边际收益。理想状态应为 100%。

### 2. 科学预期
* **Linear Speedup**: $S(m) = m$（理想情况）。
* **Sub-linear Speedup**: 由于调度开销、锁竞争、释放受限（Release-limited），实际曲线会低于理想线并逐渐趋于饱和。

In [336]:
def extract_hp_makespans(csv):
    if not os.path.exists(csv): return pd.DataFrame()
    # 读取数据，确保 t_us 为数值
    df = pd.read_csv(csv)
    df["t_us"] = pd.to_numeric(df["t_us"], errors='coerce')
    df = df.dropna(subset=["t_us"]).sort_values("t_us")

    # 1. 获取超周期边界 (main::rollover 的 release 时刻)
    # 使用 str.contains 增加容错性，防止标签有微小差异
    is_rollover = df["tag"].fillna("").str.contains("main::rollover")
    is_release = df["kind"] == "release"
    boundaries = df[is_rollover & is_release]["t_us"].sort_values().values

    if len(boundaries) < 2:
        return pd.DataFrame()

    # 2. 提取所有 Worker 任务的完成时刻
    # 我们关注任务什么时候“完工”，通常以 complete 或 finished 为准
    excl = {"timer", "main::expand", "main::rollover"}
    worker_mask = ~df["tag"].fillna("").str.contains("|".join(excl))

    # 提取 Worker 任务的 release (开始算起) 和 complete (完工)
    worker_releases = df[worker_mask & (df["kind"] == "release")][["tid", "t_us", "tag"]]
    worker_completes = df[worker_mask & (df["kind"] == "complete")][["tid", "t_us"]]

    hp_results = []

    # 3. 遍历每一个超周期窗口
    for i in range(len(boundaries) - 1):
        b_start = boundaries[i]
        b_end = boundaries[i+1]

        # 找出在本周期内释放的任务
        jobs_released = worker_releases[(worker_releases["t_us"] >= b_start) &
                                        (worker_releases["t_us"] < b_end)]

        if not jobs_released.empty:
            # 找出这些任务对应的完成时间
            # 逻辑：对于每一个在本周期 release 的任务，寻找它之后最近的一个 complete
            # 简化逻辑：在本周期内或稍后一点点时间内的所有 worker complete 事件的最大值
            # 允许 10% 的超周期溢出，以捕捉跨周期的任务完成点
            buffer = (b_end - b_start) * 0.1
            relevant_completes = worker_completes[(worker_completes["t_us"] > b_start) &
                                                  (worker_completes["t_us"] < b_end + buffer)]

            if not relevant_completes.empty:
                last_complete = relevant_completes["t_us"].max()
                makespan = last_complete - b_start

                hp_results.append({
                    "hp_index": i,
                    "makespan": makespan,
                    "job_count": len(jobs_released)
                })

    return pd.DataFrame(hp_results)

In [127]:
import re

def aggregate_speedup_by_hp(root_dir):
    # 匹配文件名，例如 trace_fork_u10_m1_s10000.csv
    pattern = re.compile(
        r"trace_(?P<kind>\w+)_u(?P<u>\d+)_m(?P<m>\d+)_s(?P<seed>\d+).*\.csv"
    )
    results = []

    if not os.path.exists(root_dir):
        print(f"目录不存在: {root_dir}")
        return pd.DataFrame()

    files = [f for f in os.listdir(root_dir) if f.endswith(".csv")]
    print(f"正在扫描: {root_dir}，找到 {len(files)} 个 CSV 文件")

    for filename in sorted(files):
        match = pattern.match(filename)
        if match:
            path = os.path.join(root_dir, filename)
            df_hp = extract_hp_makespans(path)

            if not df_hp.empty:
                median_makespan = df_hp["makespan"].median()
                results.append({
                    "kind": match.group("kind"),
                    "u": match.group("u"),
                    "m": int(match.group("m")),
                    "T_hp": median_makespan
                })
            else:
                print(f"文件 {filename} 解析结果为空 (可能是缺少 rollover 或任务)")

    if not results:
        print("没有提取到任何有效数据，请检查正则匹配或 CSV 内容。")
        return pd.DataFrame()

    agg = pd.DataFrame(results)

    # 跨 seed 取中位数
    agg_median = agg.groupby(["kind", "u", "m"])["T_hp"].median().reset_index()

    # 获取 m=1 基准
    # 注意：这里需要根据 kind 和 u 匹配
    base = agg_median[agg_median["m"] == 1].copy()
    base = base.rename(columns={"T_hp": "T1"})[["kind", "u", "T1"]]

    if base.empty:
        print("警告：未找到 m=1 的基准数据，无法计算 Speedup。")
        return agg_median

    final_df = agg_median.merge(base, on=["kind", "u"], how="left")

    # 计算加速比 S = T1 / Tm
    final_df["Speedup"] = final_df["T1"] / final_df["T_hp"]
    final_df["Efficiency"] = (final_df["Speedup"] / final_df["m"]) * 100

    return final_df.sort_values(["kind", "u", "m"])

# 执行
agg_results = aggregate_speedup_by_hp("test")
print(agg_results)

正在扫描: test，找到 270 个 CSV 文件
         kind   u  m      T_hp       T1   Speedup  Efficiency
0        fork  10  1   82529.5  82529.5  1.000000  100.000000
1        fork  10  2   59863.0  82529.5  1.378640   68.931978
2        fork  10  3   30059.5  82529.5  2.745538   91.517934
3        fork  10  4   82163.0  82529.5  1.004461   25.111516
4        fork  10  5   86372.5  82529.5  0.955507   19.110133
..        ...  .. ..       ...      ...       ...         ...
265  multihop  90  2  102138.0  48425.0  0.474113   23.705673
266  multihop  90  3  100312.0  48425.0  0.482744   16.091461
267  multihop  90  4  113685.0  48425.0  0.425958   10.648942
268  multihop  90  5  147616.5  48425.0  0.328046    6.560920
269  multihop  90  6  144617.0  48425.0  0.334850    5.580833

[270 rows x 7 columns]


## 1. 交互式加速比曲线 (Science Style)
说明：绘制 S(m) 随 m 增长的曲线，并叠加一条虚线作为 Ideal (Linear) Speedup 参考。

In [331]:
def draw_px_speedup_curve(agg_df):
    # 创建理想线性参考线数据
    m_range = agg_df["m"].unique()
    ideal_line = pd.DataFrame({"m": m_range, "Speedup": m_range, "Type": "Ideal (Linear)"})

    # 绘制实际数据
    fig = px.line(
        agg_df, x="m", y="Speedup", color="kind", symbol="kind",
        markers=True,
        title="System Speedup vs. Number of Workers",
        labels={"m": "Number of Worker Cores ($m$)", "Speedup": "Speedup $S(m)$"},
        color_discrete_sequence=[SCI_PALETTE["Active"], SCI_PALETTE["Post"]]
    )

    # 叠加理想线
    fig.add_scatter(x=ideal_line["m"], y=ideal_line["Speedup"],
                    mode='lines', name='Ideal Speedup',
                    line=dict(dash='dash', color=SCI_PALETTE["Idle"]))

    fig.update_layout(
        plot_bgcolor='white',
        xaxis=dict(gridcolor='#F0F0F0', dtick=1),
        yaxis=dict(gridcolor='#F0F0F0'),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        height=600
    )
    fig.show()

draw_px_speedup_curve(agg_results)

## 2. 并行效率分析图 (Scalability)
说明：展示 E(m) 随核心数增加的衰减情况。这是 Science 论文中用来讨论“收益递减”和“调度瓶颈”的标准图表。

In [129]:
def draw_px_efficiency_curve(agg_df):
    fig = px.line(
        agg_df, x="m", y="Efficiency", color="kind", markers=True,
        title="Parallel Efficiency Scalability",
        labels={"m": "Number of Workers ($m$)", "Efficiency": "Efficiency (%)"},
        color_discrete_sequence=[SCI_PALETTE["Overhead"], SCI_PALETTE["Wakeup"]]
    )

    # 添加 100% 效率参考线
    fig.add_hline(y=100, line_dash="dot", line_color="#000000", annotation_text="Ideal Scalability")

    fig.update_layout(
        plot_bgcolor='white',
        yaxis=dict(range=[0, 110], gridcolor='#F0F0F0'),
        xaxis=dict(gridcolor='#F0F0F0', dtick=1)
    )
    fig.show()

draw_px_efficiency_curve(agg_results)

## 3. 完工时间饱和分析 (Makespan Saturation)
说明：展示 Hp 绝对值的下降趋势。这能直观看到系统在核心数增加到多少时开始进入“平台期”。

In [130]:
def draw_px_makespan_saturation(agg_df):
    fig = px.bar(
        agg_df, x="m", y="T_hp", color="kind",
        barmode='group',
        title="Hyperperiod Makespan Reduction",
        labels={"T_hp": "Median Makespan (us)", "m": "Workers ($m$)"},
        color_discrete_sequence=[SCI_PALETTE["Active"], SCI_PALETTE["Post"]]
    )

    fig.update_layout(plot_bgcolor='white', yaxis=dict(gridcolor='#F0F0F0'))
    fig.show()

draw_px_makespan_saturation(agg_results)

### 加速比结果分析

1. **加速比饱和 (Saturation)**:
   - 如果曲线在 $m=4$ 后变平，说明系统受限于 **Amhdahl's Law**。即使核心再多，由于任务间的串行部分（如全局队列锁、主线程 `rollover`）无法并行，系统性能已达上限。

2. **效率衰减 (Efficiency Drop)**:
   - 如果 $E(m)$ 随 $m$ 增加剧烈下滑（例如从 90% 掉到 40%），说明 **调度开销 (Overhead)** 随核心数呈非线性增长。这通常是由于缓存一致性流量（Cache Coherence Traffic）或互斥锁争用导致的。

3. **释放受限下界 (Release-limited Bound)**:
   - 观察 `Makespan Reduction` 图。如果 $T_{hp}$ 最终接近最长算法节点的单次执行时间，说明你已经消除了所有调度空隙，达到了物理极限。